# RAG with Docling and LangChain

| Step | Tech | Execution |
| --- | --- | --- |
| Embedding | OpenRouter Embeddings API (free model) | 🌐 Remote |
| Vector store | InMemoryVectorStore (optional Chroma) | 💻 Local |
| Gen AI | Hugging Face Inference API + OpenRouter Chat | 🌐 Remote |

This example leverages the
[LangChain Docling integration](../../integrations/langchain/), OpenRouter embeddings,
and an in-memory vector store for a fast local demo loop.

The presented `DoclingLoader` component enables you to:
- use various document types in your LLM applications with ease and speed, and
- leverage Docling's rich format for advanced, document-native grounding.

`DoclingLoader` supports two different export modes:
- `ExportType.MARKDOWN`: if you want to capture each input document as a separate
  LangChain document, or
- `ExportType.DOC_CHUNKS` (default): if you want to have each input document chunked and
  to then capture each individual chunk as a separate LangChain document downstream.

The example allows exploring both modes via parameter `EXPORT_TYPE`; depending on the
value set, the example pipeline is then set up accordingly.

## Setup

- 👉 For best conversion speed, use GPU acceleration whenever available; e.g. if running on Colab, use GPU-enabled runtime.
- Notebook uses HuggingFace's Inference API; for increased LLM quota, token can be provided via env var `HF_TOKEN`.
- Requirements can be installed as shown below (`--no-warn-conflicts` meant for Colab's pre-populated Python env; feel free to remove for stricter usage):

In [4]:
%pip install -q --progress-bar off --no-warn-conflicts \
    langchain-docling langchain-core langchain-huggingface \
    langchain-openai langgraph langchain python-dotenv rich requests

  Preparing metadata (setup.py) ... done


In [ ]:
import json
import os
from textwrap import dedent
from rich import print

In [ ]:
from dotenv import load_dotenv

def _get_env_from_colab_or_os(key: str) -> str | None:
    """Read an environment variable from Colab secrets first, then OS env."""
    try:
        from google.colab import userdata
        try:
            return userdata.get(key)
        except userdata.SecretNotFoundError:
            pass
    except ImportError:
        pass
    return os.getenv(key)

In [8]:
load_dotenv()

# Disable tokenizer parallel workers to avoid noisy notebook warnings.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [9]:
def clip_text(text: str, threshold: int = 100) -> str:
    """Return a truncated preview of long text for cleaner notebook output."""
    return f"{text[:threshold]}..." if len(text) > threshold else text

### Observation

If setup succeeded, environment variables are loaded, helper utilities are available, and model/provider IDs are configured for both Hugging Face and OpenRouter usage.

## Document loading

Now we can instantiate our loader and load documents.

This section loads and chunks the source document using Docling so we can build retrieval units for the RAG pipeline.

In [10]:
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

from transformers import AutoTokenizer


# Source document for the lesson; small and publicly available for reproducible demos.
file_path = ["https://arxiv.org/pdf/2408.09869"]

# Same as embedding model.
embed_model_id = "sentence-transformers/all-MiniLM-L6-v2"

# Export mode used by this lesson; DOC_CHUNKS gives retrieval-ready chunks directly.
export_type = ExportType.DOC_CHUNKS

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(embed_model_id)
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

In [11]:
# from docling.datamodel.pipeline_options import PdfPipelineOptions

# pipeline_options = PdfPipelineOptions()
# enable third-party plugins (like langchain_docling)
# pipeline_options.allow_external_plugins = True  

loader = DoclingLoader(
    file_path=file_path,
    export_type=export_type,
    chunker=HybridChunker(tokenizer=tokenizer),
    # convert_kwargs={"pipeline_options": pipeline_options},
)

docs = loader.load()

[INFO] 2026-04-28 14:48:41,572 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-28 14:48:41,578 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-28 14:48:41,584 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-28 14:48:43,115 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-04-28 14:48:43,352 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-28 14:48:43,354 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-28 14:48:43,683 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-28 14:48:43,684 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-28 14:48:43,688 [RapidOCR] download_file.py:68: Initiating download: https

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (638 > 512). Running this sequence through the model will result in indexing errors


In [63]:
loader

In [ ]:
(type(docs), len(docs))

(list, 50)

In [29]:
print(docs[1].page_content)

Abstract
This technical report introduces Docling , an easy to use, self-contained, MITlicensed open-source package for PDF 
document conversion. It is powered by state-of-the-art specialized AI models for layout analysis (DocLayNet) and 
table structure recognition (TableFormer), and runs efficiently on commodity hardware in a small resource budget. 
The code interface allows for easy extensibility and addition of new features and models.

In [30]:
print(docs[2].page_content)

1 Introduction
Converting PDF documents back into a machine-processable format has been a major challenge for decades due to their
huge variability in formats, weak standardization and printing-optimized characteristic, which discards most 
structural features and metadata. With the advent of LLMs and popular application patterns such as 
retrieval-augmented generation (RAG), leveraging the rich content embedded in PDFs has become ever more relevant. 
In the past decade, several powerful document understanding solutions have emerged on the market, most of which are
commercial software, cloud offerings [3] and most recently, multi-modal vision-language models. As of today, only a
handful of open-source tools cover PDF conversion, leaving a significant feature and quality gap to proprietary 
solutions.
With Docling , we open-source a very capable and efficient document conversion tool which builds on the powerful, 
specialized AI models and datasets for layout analysis and table structure recognition we developed and presented 
in the recent past [12, 13, 9]. Docling is designed as a simple, self-contained python library with permissive 
license, running entirely locally on commodity hardware. Its code architecture allows for easy extensibility and 
addition of new features and models.
Here is what Docling delivers today:

In [31]:
print(docs[3].page_content)

1 Introduction
- Converts PDF documents to JSON or Markdown format, stable and lightning fast
- Understands detailed page layout, reading order, locates figures and recovers table structures
- Extracts metadata from the document, such as title, authors, references and language
- Optionally applies OCR, e.g. for scanned PDFs
- Can be configured to be optimal for batch-mode (i.e high throughput, low time-to-solution)
- or interactive mode (compromise on efficiency, low time-to-solution)
- Can leverage different accelerators (GPU, MPS, etc).

In [32]:
print(docs[4].page_content)

2 Getting Started
To use Docling, you can simply install the docling package from PyPI. Documentation and examples are available in 
our GitHub repository at github.com/DS4SD/docling. All required model assets 1 are downloaded to a local 
huggingface datasets cache on first use, unless you choose to pre-install the model assets in advance.
Docling provides an easy code interface to convert PDF documents from file system, URLs or binary streams, and 
retrieve the output in either JSON or Markdown format. For convenience, separate methods are offered to convert 
single documents or batches of documents. A basic usage example is illustrated below. Further examples are 
available in the Doclign code repository.
from docling.document_converter import DocumentConverter

In [33]:
print(docs[5].page_content)

2 Getting Started
```
source = "https://arxiv.org/pdf/2206.01062" # PDF path or URL converter = DocumentConverter() result = 
converter.convert_single(source) print(result.render_as_markdown()) # output: "## DocLayNet: A Large Human 
-Annotated Dataset for Document -Layout Analysis [...]"
```
Optionally, you can configure custom pipeline features and runtime options, such as turning on or off features 
(e.g. OCR, table structure recognition), enforcing limits on the input document size, and defining the budget of 
CPU threads. Advanced usage examples and options are documented in the README file. Docling also provides a 
Dockerfile to demonstrate how to install and run it inside a container.

In [ ]:
# if export_type == ExportType.DOC_CHUNKS:
#     splits = docs
# elif export_type == ExportType.MARKDOWN:
#     from langchain_text_splitters import MarkdownHeaderTextSplitter

#     splitter = MarkdownHeaderTextSplitter(
#         headers_to_split_on = [
#             ("#", "Header 1"),
#             ("##", "Header 2"),
#             ("###", "Header 3"),
#         ],
#     )
#     splits = [split for doc in docs for split in splitter.split_text(doc.page_content)]
# else:
#     raise ValueError(f"Unexpected export type: {export_type}")

## Ingestion

This section builds embeddings with OpenRouter's free embedding model and ingests document chunks into `InMemoryVectorStore`. A commented Chroma option is included for easy persistence later.

We will be using HuggingFace:

In [35]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name=embed_model_id)

vectorstore = InMemoryVectorStore.from_documents(
    documents=docs,
    embedding=embedding,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Realistically, we would want to use a Vector Database like `chromadb`. But for quick testing, we will stick to `InMemoryVectorStore`:

In [ ]:
# from langchain_chroma import Chroma

# vectorstore = Chroma.from_documents(
#     documents=docs,
#     embedding=embedding,
#     collection_name="docling_demo",
# )

Uncomment the following block to use an Embedding model from OpenRouter:

In [ ]:
# import requests

# from langchain_core.embeddings import Embeddings

# class EmbeddingsAdapter(Embeddings):
#     """Minimal LangChain Embeddings adapter backed by OpenRouter's embeddings API."""

#     def __init__(self, api_key: str, model: str):
#         self.api_key = api_key
#         self.model = model
#         self.url = "https://openrouter.ai/api/v1/embeddings"

#     def _embed(self, texts: list[str]) -> list[list[float]]:
#         response = requests.post(
#             self.url,
#             headers={
#                 "Authorization": f"Bearer {self.api_key}",
#                 "Content-Type": "application/json",
#             },
#             json={"model": self.model, "input": texts},
#             timeout=60,
#         )
#         response.raise_for_status()
#         data = response.json()["data"]
#         return [row["embedding"] for row in data]

#     def embed_documents(self, texts: list[str]) -> list[list[float]]:
#         """Embed multiple documents in one request."""
#         return self._embed(texts)

#     def embed_query(self, text: str) -> list[float]:
#         """Embed a single query string."""
#         return self._embed([text])[0]

# # Required credential for remote embedding requests to OpenRouter.
# openrouter_api_key = _get_env_from_colab_or_os("OPENROUTER_API_KEY")

# # Free embedding model used in the lesson to avoid paid API usage.
# embed_model_id = "nvidia/llama-nemotron-embed-vl-1b-v2:free"

# embedding = EmbeddingsAdapter(
#     api_key=openrouter_api_key,
#     model=embed_model_id,
# )

## RAG

### Retriever

In [36]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

### LLM

In [ ]:
from langchain_openai import ChatOpenAI

# OpenRouter chat model
openrouter_llm = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=_get_env_from_colab_or_os("OPENROUTER_API_KEY"),
)

Alternatively, one could use HuggingFace servers and run an open-source model there:

In [ ]:
# Hugging Face Inference API integrated directly as a LangChain model.
# from langchain_huggingface import HuggingFaceEndpoint

# hf_llm = HuggingFaceEndpoint(
#     repo_id="mistralai/Mixtral-8x7B-Instruct-v0.1",
#     huggingfacehub_api_token=_get_env_from_colab_or_os("HF_TOKEN"),
#     task="text-generation",
# )

In [39]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.func import entrypoint, task

In [40]:
@task
def retrieve(query: str):
    """Fetch top-k relevant documents from the vector store retriever."""
    return retriever.invoke(query)

In [41]:
@task
def generate(query: str, passages: list[str], use_openrouter: bool = True) -> str:
    """Generate an answer from retrieved passages using a selected LangChain LLM."""
    system_text = dedent("""
        Role: You are a helpful and informative assistant. Answer using the reference passages below.
        Style and Tone: Respond in complete sentences for a non-technical audience in a friendly, conversational tone.
        Note: If a passage is irrelevant, you may ignore it.
    """)
    passages_serialized = "\n".join(f"<passage>{p}</passage>" for p in passages)
    user_text = dedent(f"""
        <question>{query}</question>
        <passages>
        {passages_serialized}
        </passages>
    """)

    if use_openrouter:
        msg = openrouter_llm.invoke([
            SystemMessage(system_text),
            HumanMessage(user_text)
        ])
        return msg.content

    # prompt = f"{system_text}\n\n{user_text}"
    # return hf_llm.invoke(prompt)

In [ ]:
@entrypoint()
def rag_qa_workflow(query: str, use_openrouter: bool = True) -> dict:
    """Run retrieval and generation as a functional LangGraph RAG workflow."""
    docs = retrieve(query).result()
    answer = generate(query, [doc.page_content for doc in docs], use_openrouter).result()
    return {"question": query, "answer": answer, "context": docs}

This cell executes the functional RAG workflow end-to-end (retrieve then generate), prints a clipped final answer, and shows the source chunks used for grounding.

In [48]:
# Demo query chosen to match the Docling report section headers for clear retrieval traces.
question = "Which are the main AI models in Docling?"
d = rag_qa_workflow.invoke(question, use_openrouter=True)

In [51]:
clipped_answer = clip_text(d["answer"], threshold=200)
print(f"Question:\n{d['question']}\n\nAnswer:\n{clipped_answer}")
for i, doc in enumerate(d["context"]):
    print()
    print(f"Source {i + 1}:")
    print(f"  text: {json.dumps(clip_text(doc.page_content, threshold=350))}")
    for key in doc.metadata:
        if key != "pk":
            val = doc.metadata.get(key)
            clipped_val = clip_text(val) if isinstance(val, str) else val
            print(f"  {key}: {clipped_val}")

Question:
Which are the main AI models in Docling?

Answer:
The mainAI models that Docling ships are:

1. **A layout‑analysis model** – it works like an object detector, identifying and locating different page elements
(such as headings, paragraphs, images, et...

Source 1:

text: "3.2 AI models\nAs part of Docling, we initially release two highly capable AI models to the open-source 
community, which have been developed and published recently by our team. The first model is a layout analysis 
model, an accurate object-detector for page elements [13]. The second model is TableFormer [12, 9], a 
state-of-the-art table structure re..."

source: https://arxiv.org/pdf/2408.09869

dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': 
[{'self_ref': '#/texts/50', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text',
'prov': [{'page_no': 3, 'bbox': {'l': 108.0, 't': 404.8726526000001, 'r': 504.00338739999984, 'b': 
330.86577439292046, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 608]}]}], 'headings': ['3.2 AI models'], 
'origin': {'mimetype': 'application/pdf', 'binary_hash': 11465328351749295394, 'filename': '2408.09869'}}

...

dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': 
[{'self_ref': '#/texts/27', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text',
'prov': [{'page_no': 2, 'bbox': {'l': 108.0, 't': 272.7486525999999, 'r': 504.0033873999997, 'b': 
176.92377439292034, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 796]}]}], 'headings': ['3 Processing pipeline'],
'origin': {'mimetype': 'application/pdf', 'binary_hash': 11465328351749295394, 'filename': '2408.09869'}}

Source 3:

text: "6 Future work and contributions\nDocling is designed to allow easy extension of the model library and 
pipelines. In the future, we plan to extend Docling with several more models, such as a figure-classifier model, an
equationrecognition model, a code-recognition model and more. This will help improve the quality of conversion for 
specific types of ..."

source: https://arxiv.org/pdf/2408.09869

dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': 
[{'self_ref': '#/texts/75', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text',
'prov': [{'page_no': 5, 'bbox': {'l': 108.00000000000006, 't': 322.1996526000001, 'r': 504.00338739999984, 'b': 
259.10277439292054, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 543]}]}, {'self_ref': '#/texts/76', 'parent': 
{'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 5, 'bbox': {'l':
108.00000000000006, 't': 251.6541940000002, 'r': 504.00338739999967, 'b': 199.07777439292056, 'coord_origin': 
'BOTTOMLEFT'}, 'charspan': [0, 402]}]}], 'headings': ['6 Future work and contributions'], 'origin': {'mimetype': 
'application/pdf', 'binary_hash': 11465328351749295394, 'filename': '2408.09869'}}